In [4]:
import pandas as pd
from difflib import SequenceMatcher

In [5]:
# Add the csv files you want to compare here
file1 = './UIB_LUBER.csv'
file2 = './UIB_LUBER2.csv'
output_file = './learning_outcome_differences.csv'

In [6]:
# Function to compare two strings and return the differences as a string
def compare_strings(str1, str2):
    seq_match = SequenceMatcher(None, str1, str2)
    # Find matching blocks and calculate the differing parts
    diffs = []
    blocks = seq_match.get_matching_blocks()

    # Extract the differing sections from the original strings
    start = 0
    for block in blocks:
        # Extract non-matching text before the current match
        diff1 = str1[start:block[0]]
        diff2 = str2[start:block[1]]
        if diff1:
            diffs.append(f"Only in first file: {diff1}")
        if diff2:
            diffs.append(f"Only in second file: {diff2}")
        start = block[0] + block[2]  # Move the starting point after the current match
    
    return diffs if diffs else None  # If no differences, return None

# Function to compare the two CSV files
def compare_csv_files(file1, file2, output_file):
    # Read the CSV files into DataFrames
    df1 = pd.read_csv(file1)
    df2 = pd.read_csv(file2)

    # Group by school, study program, and type of learning outcome
    df1_grouped = df1.groupby(['Skole', 'Studie program', 'Læringsutbytte type']).agg(lambda x: ' '.join(x)).reset_index()
    df2_grouped = df2.groupby(['Skole', 'Studie program', 'Læringsutbytte type']).agg(lambda x: ' '.join(x)).reset_index()

    # Initialize an empty list to store the differences
    differences = []

    # Iterate over the rows of the first DataFrame
    for index, row in df1_grouped.iterrows():
        school = row['Skole']
        study_program = row['Studie program']
        type_of_learning_outcome = row['Læringsutbytte type']
        learning_outcome_1 = row['Læringsutbytte']  # Learning outcome text

        # Check if the corresponding row exists in the second DataFrame
        matching_row = df2_grouped[(df2_grouped['Skole'] == school) & 
                                   (df2_grouped['Studie program'] == study_program) & 
                                   (df2_grouped['Læringsutbytte type'] == type_of_learning_outcome)]

        if not matching_row.empty:
            learning_outcome_2 = matching_row.iloc[0]['Læringsutbytte']  # Extract the learning outcome text

            # Compare the learning outcomes using SequenceMatcher
            diff = compare_strings(learning_outcome_1, learning_outcome_2)

            # If there are differences, append to the differences list
            if diff is not None:
                differences.append({
                    'Skole': school,
                    'Studie program': study_program,
                    'Læringsutbytte type': type_of_learning_outcome,
                    'Læringsutbytte first file': learning_outcome_1,
                    'Læringsutbytte second file': learning_outcome_2,
                    'Differences': '; '.join(diff)
                })
        else:
            differences.append({
                'Skole': school,
                'Studie program': study_program,
                'Læringsutbytte type': type_of_learning_outcome,
                'Læringsutbytte first file': learning_outcome_1,
                'Læringsutbytte second file': 'Missing in second file',
                'Differences': 'Missing in second file'
            })

    # Check if no differences were found
    if not differences:
        print("No differences found between the files.")
    else:
        # Create a DataFrame from the differences
        diff_df = pd.DataFrame(differences)

        # Write the differences to a new CSV file
        diff_df.to_csv(output_file, index=False)
        print(f"Differences have been written to {output_file}")
        

# The dofferences between the files will be written to learning_outcomes_differences.csv
compare_csv_files(file1, file2, output_file)

Differences have been written to ./learning_outcome_differences.csv
